In [1]:
!pip install -q tiktoken

In [2]:
from pathlib import Path
import sys
import os
import torch

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/arjunbhattarai123/"
    "gpt-2-llm-from-scratch-training-project-1/LLMs"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_ROOT}"
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "gpt2_wikitext.pt"
)

print("Project root:", PROJECT_ROOT)
print("Device:", device)
print("Checkpoint exists:", CHECKPOINT_PATH.exists())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/LLMs
Device: cuda
Checkpoint exists: True
GPU: Tesla T4


In [3]:
train_path = (
    PROJECT_ROOT
    / "data"
    / "wikitext-103"
    / "train.txt"
)

validation_path = (
    PROJECT_ROOT
    / "data"
    / "wikitext-103"
    / "validation.txt"
)

train_text = train_path.read_text(encoding="utf-8")
validation_text = validation_path.read_text(encoding="utf-8")

print("Training characters:", len(train_text))
print("Validation characters:", len(validation_text))

Training characters: 540624389
Validation characters: 1147070


In [4]:
import tiktoken

from llm.preprocessing.dataset import create_data_loader

tokenizer = tiktoken.get_encoding("gpt2")

context_length = 1024

train_loader = create_data_loader(
    train_text,
    batch_size=1,
    max_length=context_length,
    stride=context_length,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

validation_loader = create_data_loader(
    validation_text,
    batch_size=1,
    max_length=context_length,
    stride=context_length,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Training batches: 116294
Validation batches: 243


In [5]:
import torch

from llm.gpt.config import GPT_CONFIG_124M
from llm.gpt.model import GPTModel

config = GPT_CONFIG_124M.copy()
config["context_length"] = context_length

model = GPTModel(config).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.1,
)

In [6]:
from llm.training.trainer import train_model_simple

OUTPUT_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_20k.pt"
)

train_losses, validation_losses, tokens_seen = (
    train_model_simple(
        model=model,
        train_loader=train_loader,
        val_loader=validation_loader,
        optimizer=optimizer,
        device=device,

        # Must be greater than the saved epoch of 4.
        num_epochs=100,

        eval_freq=500,
        eval_iter=5,
        start_context="The history of the world",
        tokenizer=tokenizer,

        checkpoint_path=str(OUTPUT_CHECKPOINT),
        config=config,

        # This is the total target, not additional steps.
        max_steps=20_000,

        # Restore model, optimizer, epoch, and AMP scaler.
        resume_from_checkpoint=str(CHECKPOINT_PATH),
    )
)

AMP enabled: FP16

Resuming from checkpoint: /kaggle/input/datasets/arjunbhattarai123/gpt-2-llm-from-scratch-training-project-1/LLMs/checkpoints/gpt2_wikitext.pt
No AMP GradScaler state found in checkpoint. Using a fresh scaler.
Resumed from step: 9999
Resumed from epoch: 4
Ep 5 (Step 010000): Train loss 2.957, Val loss 2.839
Ep 5 (Step 010500): Train loss 3.101, Val loss 2.837
Ep 5 (Step 011000): Train loss 2.761, Val loss 2.836
Ep 5 (Step 011500): Train loss 3.016, Val loss 2.833
Ep 5 (Step 012000): Train loss 3.055, Val loss 2.824
Ep 5 (Step 012500): Train loss 2.851, Val loss 2.826
Ep 5 (Step 013000): Train loss 3.030, Val loss 2.826
Ep 5 (Step 013500): Train loss 2.987, Val loss 2.823
Ep 5 (Step 014000): Train loss 3.028, Val loss 2.821
Ep 5 (Step 014500): Train loss 2.886, Val loss 2.816
Ep 5 (Step 015000): Train loss 2.944, Val loss 2.815
Ep 5 (Step 015500): Train loss 3.097, Val loss 2.812
Ep 5 (Step 016000): Train loss 2.925, Val loss 2.810
Ep 5 (Step 016500): Train loss 2.994

In [7]:
from pathlib import Path
import torch

OUTPUT_CHECKPOINT = Path(
    "/kaggle/working/gpt2_wikitext_20k.pt"
)

print("Exists:", OUTPUT_CHECKPOINT.exists())

if OUTPUT_CHECKPOINT.exists():
    print(
        "Size:",
        round(OUTPUT_CHECKPOINT.stat().st_size / 1024**3, 2),
        "GB",
    )

    checkpoint = torch.load(
        OUTPUT_CHECKPOINT,
        map_location="cpu",
    )

    print("Global step:", checkpoint["global_step"])
    print("Epoch:", checkpoint["epoch"])
    print("Training loss:", checkpoint["train_loss"])
    print("Validation loss:", checkpoint["val_loss"])

Exists: True
Size: 1.87 GB
Global step: 19999
Epoch: 5
Training loss: 3.271234226226807
Validation loss: 2.790156030654907
